In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

root = "logs"
fig_save_folder = os.path.join("figs", "initial_entropy")

# Initial progress of Actor-Critic on Tomography

In `2026_06_26/exp_0`, we ran just one seed of our initial implementation of RL (actor-critic) onto a Tomography problem. We considered:
- `exp_0`: PNSR_INCREMENTAL reward with `entropy_coeff=0.0`
- `exp_1`: PNSR_INCREMENTAL reward with `entropy_coeff=-0.01`
- `exp_2`: FWD_INCREMENTAL reward with `entropy_coeff=0.0`
- `exp_3`: FWD_INCREMENTAL reward with `entropy_coeff=-0.01`

In [ ]:
max_ep = 30_000
n_seeds = 1
seed_offset = 1029
n_exps = 4
save_figs = True

In [ ]:
n_eps_arr = np.zeros((n_exps, n_seeds), dtype=int)
episode = np.zeros((n_exps, n_seeds, max_ep), dtype=int)
reward = np.zeros((n_exps, n_seeds, max_ep), dtype=float)
entropy = np.zeros((n_exps, n_seeds, max_ep), dtype=float)
l_1_dist = np.zeros((n_exps, n_seeds, max_ep), dtype=float)
exp_id = 0

for i in range(n_exps):
    for k in range(n_seeds):
        j = seed_offset + k
        fname = os.path.join(root, "2026_06_26", "exp_0", "run_%d" % i, "seed=%d.csv" % j)
        df = pd.read_csv(fname, header="infer")
    
        n_eps_arr[i,k] = len(df)
        episode[i,k,:n_eps_arr[i,k]] = df['episode']
        reward[i,k,:n_eps_arr[i,k]] = df['episodic reward']
        entropy[i,k,:n_eps_arr[i,k]] = df['entropy']
        l_1_dist[i,k,:n_eps_arr[i,k]] = df['l_1']

### Plot rewards 

In [ ]:
plt.style.use('ggplot')
_, axes = plt.subplots(ncols=2, figsize=(7,4))
label_arr = ["PNSR_INC", "FWD_INC"]
sub_label_arr = ["0.0", "-0.01"]
color_arr = ["black", "red"]
lss_arr = ["solid", "dotted"]
k = 0 # seed
avg_len = 30
avg_arr = np.ones(avg_len, dtype=float)/avg_len

for i in range(n_exps):
    xs = episode[i,k,:n_eps_arr[i,k]][avg_len-1:-(avg_len-1)]
    ys = np.convolve(reward[i,k,:n_eps_arr[i,k]] , avg_arr, mode="same")[avg_len-1:-(avg_len-1)]
    axes[i//2].plot(xs, ys, 
                    label="Entropy: %s" % sub_label_arr[i%2], 
                    color=color_arr[i%2],
                    linestyle=lss_arr[i%2],
                   )

axes[0].set(
    ylabel="Reward smoothed(30) (higher is better)",
    xlabel="Episodes (6 xrays/ep)",
    title="RL online %s reward" % label_arr[0],
)
axes[1].set(
    xlabel="Episodes (6 xrays/ep)",
    title="RL online %s reward" % label_arr[1],
)
axes[0].legend()

if save_figs:
    fname = os.path.join(fig_save_folder, "reward.png")
    plt.savefig(fname, dpi=180)

### Plot distance to uniform
We will measure entropy of policies encountered during the algorithm.

$$
H(p) := -\sum_{a \in \mathrm{angles}} p(a) \ln(a).
$$

The higher $H(\cdot)$, the closer we are to a uniform policy.

In [ ]:
plt.style.use('ggplot')
_, axes = plt.subplots(ncols=2, figsize=(7,4))
label_arr = ["PNSR_INC", "FWD_INC"]
sub_label_arr = ["0.0", "-0.01"]
color_arr = ["black", "red"]
lss_arr = ["solid", "dotted"]
k = 0 # seed

for i in range(n_exps):
    xs = episode[i,k,:n_eps_arr[i,k]]
    ys = entropy[i,k,:n_eps_arr[i,k]] 
    axes[i//2].plot(xs, ys, 
                    label="Entropy: %s" % sub_label_arr[i%2], 
                    color=color_arr[i%2],
                    linestyle=lss_arr[i%2],
                   )

axes[0].set(
    ylabel="Entropy (higher is closer to Uni)",
    xlabel="Episodes (6 xrays/ep)",
    title="RL online %s entropy" % label_arr[0],
)
axes[1].set(
    xlabel="Episodes (6 xrays/ep)",
    title="RL online %s entropy" % label_arr[1],
)
axes[0].legend()

if save_figs:
    fname = os.path.join(fig_save_folder, "entropy.png")
    plt.savefig(fname, dpi=180)

We will measure the $\ell_1$ distance to uniformity,

$$
d_1(p,Uni) := \sum_{a \in \mathrm{angles}} \Big \vert p(a) - \frac{1}{\vert \mathrm{angles} \vert} \Big \vert.
$$

The lower $d_1(\cdot,Uni)$, the closer we are to a uniform policy.

In [ ]:
plt.style.use('ggplot')
_, axes = plt.subplots(ncols=2, figsize=(7,4))
label_arr = ["PNSR_INC", "FWD_INC"]
sub_label_arr = ["0.0", "-0.01"]
color_arr = ["black", "red"]
lss_arr = ["solid", "dotted"]
k = 0 # seed

for i in range(n_exps):
    xs = episode[i,k,:n_eps_arr[i,k]]
    ys = l_1_dist[i,k,:n_eps_arr[i,k]] 
    axes[i//2].plot(xs, ys, 
                    label="Entropy: %s" % sub_label_arr[i%2], 
                    color=color_arr[i%2],
                    linestyle=lss_arr[i%2],
                   )

axes[0].set(
    ylabel=r"$\ell_1$-dist (lower is closer to Uni)",
    xlabel="Episodes (6 xrays/ep)",
    title=r"RL online %s $\ell_1$-dist" % label_arr[0],
)
axes[1].set(
    xlabel="Episodes (6 xrays/ep)",
    title=r"RL online %s $\ell_1$-dist" % label_arr[1],
)
axes[0].legend()

if save_figs:
    fname = os.path.join(fig_save_folder, "l1_dist.png")
    plt.savefig(fname, dpi=180)